# v136 — 2-Fold Honest (471 観測) 強検出器 1対1 連続融合 & 交差盲検セレクタ

`v132` で Fold-0 側の足を引っ張っていた `croptrain` (`mAP50=0.598`) を外し、漏洩なしの強検出器群を統合評価します：
- **`prod`**: `yolo_fold_0` / `yolo_fold_2`
- **`yolo26`**: `v121_yolo26_f0` / `v117_yolo26`
- **`small_tal`**: `v133a` / `v133b` (`artifacts/v133_oof_caches` 自動再利用)
- **`spine_small_tal`**: `v134a` / `v134b` (`artifacts/v134_oof_caches` が揃っていれば自動で 4 検出器モードに拡張、学習中なら 3 検出器モードで即実行可能)

`prod` と `yolo26` の推論結果も `artifacts/v136_oof_caches/` に永続化するため、2 回目以降（`v134` 完走後の再実行など）は **GPU 推論ゼロ・約 2 分** で完了します。


In [ ]:
# [1/4] 環境・Drive マウント・キャッシュ/重み確認
%pip install -q "ultralytics==8.4.120" pycocotools
import hashlib, json, math, os, pickle, shutil, tempfile, time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from pycocotools import mask as mask_utils
from tqdm.auto import tqdm
from ultralytics import YOLO

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/solar_filament_v30")
DATA_CACHE = DRIVE_ROOT / "competition_data"
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
PREPROC_CACHE = DRIVE_ROOT / "preprocessed_v30_q96"
LOCAL_PREPROC = Path("/content/solar_v136/preprocessed")
LOCAL_PREPROC.mkdir(parents=True, exist_ok=True)

V136_CACHE_DIR = ARTIFACT_ROOT / "v136_oof_caches"
V136_CACHE_DIR.mkdir(parents=True, exist_ok=True)
V133_CACHE_DIR = ARTIFACT_ROOT / "v133_oof_caches"
V134_CACHE_DIR = ARTIFACT_ROOT / "v134_oof_caches"
V138_CACHE_DIR = ARTIFACT_ROOT / "v138_oof_caches"

GROUPED_FOLDS_SHA256 = "febb3f67237a058fa8a71f188fac262035048b55d860c0e4509514877f6af67e"
FOLDS = (0, 2)

BASE_CKPTS = {
    0: {
        "prod": DRIVE_ROOT / "runs" / "yolo_fold_0" / "weights" / "best.pt",
        "yolo26": DRIVE_ROOT / "runs" / "v121_yolo26_f0" / "yolo_yolo26m_fold0" / "weights" / "best.pt",
        "refiner": ARTIFACT_ROOT / "mediseg_edgeattnet_refiner_fold0_resume.pt",
    },
    2: {
        "prod": DRIVE_ROOT / "runs" / "yolo_fold_2" / "weights" / "best.pt",
        "yolo26": DRIVE_ROOT / "runs" / "v117_yolo26" / "yolo_yolo26m_fold2" / "weights" / "best.pt",
        "refiner": ARTIFACT_ROOT / "mediseg_edgeattnet_refiner_fold2_resume.pt",
    },
}

for f, mp in BASE_CKPTS.items():
    for k, pt in mp.items():
        assert pt.exists(), f"Missing fold {f} {k}: {pt}"

has_v133_best = all((V133_CACHE_DIR / f"small_tal_f{f}_best_cache.pkl").exists() for f in FOLDS)
has_v133_last = all((V133_CACHE_DIR / f"small_tal_f{f}_last_cache.pkl").exists() for f in FOLDS)
has_v134_best = all((V134_CACHE_DIR / f"spine_small_tal_f{f}_best_cache.pkl").exists() for f in FOLDS)
has_v134_last = all((V134_CACHE_DIR / f"spine_small_tal_f{f}_last_cache.pkl").exists() for f in FOLDS)
has_v138_best = all((V138_CACHE_DIR / f"spine_aux_f{f}_best_cache.pkl").exists() for f in FOLDS)
has_v138_last = all((V138_CACHE_DIR / f"spine_aux_f{f}_last_cache.pkl").exists() for f in FOLDS)

assert has_v133_best, "v133a/v133b の OOF キャッシュがまだ揃っていません (v133_oof_caches)"
print({
    "has_v133_best": has_v133_best,
    "has_v133_last": has_v133_last,
    "has_v134_best": has_v134_best,
    "has_v134_last": has_v134_last,
    "has_v138_best": has_v138_best,
    "has_v138_last": has_v138_last,
})


In [ ]:
# [2/4] MediSegEdgeAttUNet & 1-to-1 Consensus Fusion モジュール定義
"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)

# ============================================================
# 4. Edge-Guided Attention U-Net (EG-MHSA)
# ============================================================
# 2026-08-29、WINNING_STRATEGY_2026-08-29.md §1.4のPhase 2実験用に追加。
# 参考文献: "EdgeAttNet: Towards Barb-Aware Filament Segmentation"
# (arXiv:2509.02964)。MAGFiLO上でU-Net(mIoU_pairwise 0.5724)から
# EdgeAttNet(0.6451)への改善を報告しており、barb(フィラメントの微細な
# 枝構造)の認識精度向上を狙ったアーキテクチャ。本プロジェクトの
# refiner(MediSegUNet)が抱える主要な弱点(near-miss境界品質)と直接
# 対応するため、置き換え候補として追試する。
#
# 【重要な注意】著者の公開リポジトリ(github.com/dasjar/EdgeAttNet)を
# 直接確認したところ、`model/edgeattnet_model.py`(本来UNetEdgeTransformer
# クラスが定義されているべきファイル)の中身が誤って`main.py`と同一に
# なっており、実際のアーキテクチャコードはリポジトリ上に存在しない
# (2026-08-29、`gh api`でリポジトリ全ファイルを走査して確認)。学習済み
# 重みの配布も確認できなかった。**以下は論文本文のアーキテクチャ記述
# (edge prior E=σ(Conv3x3(入力画像))をbottleneckへbilinear補間し
# 1x1convで射影、Q・Kへ加算、位置エンコーディングなし)のみを根拠にした
# 独自の再実装であり、著者オリジナル実装とのコード上の一致は未検証。**
# ベースにするU-Net本体はMediSegUNetをそのまま流用しているため、
# パラメータ総数は論文記載の値(22.6M)とは一致しない。


class EdgeGuidedMHSA(nn.Module):
    """Edge-Guided Multi-Head Self-Attention (EG-MHSA)。

    U-NetのbottleneckでSelf-Attentionを行う際、入力画像から抽出した
    エッジ事前分布をQuery/Keyに加算し、境界・barb構造への感度を高める。
    Value・出力チャネル数はbottleneckの特徴マップと同一に保つ
    (MediSegUNetへdrop-inできるよう設計)。
    """

    def __init__(self, in_channels: int, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        # E = sigmoid(Conv3x3(入力画像)): 単一チャンネルのエッジ事前分布
        self.edge_conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)
        # E' = Conv1x1(interpolate(E)): bottleneckのチャンネル数へ射影
        self.edge_proj = nn.Conv2d(1, bottleneck_channels, kernel_size=1)
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor, original_image: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        edge_map = torch.sigmoid(self.edge_conv(original_image))
        edge_map = F.interpolate(edge_map, size=(h, w), mode="bilinear", align_corners=True)
        edge_prior = self.edge_proj(edge_map)  # (B, C, H, W)

        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        edge_flat = edge_prior.flatten(2).transpose(1, 2)  # (B, N, C)
        qk = x_flat + edge_flat  # Q = K = x + E'。位置エンコーディングは使わない(論文通り)

        attn_out, _ = self.mhsa(qk, qk, x_flat)  # V = x(エッジ由来の加算なし)
        attn_out = self.norm(attn_out + x_flat)  # 残差接続 + LayerNorm(論文に明記はないが標準的なTransformerブロックとして採用)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegEdgeAttUNet(nn.Module):
    """MediSegUNetのbottleneckにEdgeGuidedMHSAを挿入したdrop-in代替。

    入出力インターフェースはMediSegUNetと完全に同一(in_channels=3の
    crop、out_channels=1の確率マップ)。既存のrefiner学習・推論コードを
    変更せず、モデルクラスの差し替えだけで比較できるように設計している。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.edge_attn = EdgeGuidedMHSA(in_channels, bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.edge_attn(x5, x)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


# ============================================================
# 5. Plain MHSA U-Net (matched-parameter ablation control, no edge prior)
# ============================================================
# 2026-08-31、report/main.tex Future Work item 1(matched-parameter
# ablation)のために追加。EdgeGuidedMHSAからedge_conv/edge_projを除いた
# だけの構成——Q=K=V=bottleneck特徴量の標準的なself-attentionで、
# edge priorをQ/Kへ加算する項だけがない。edge_conv/edge_projは
# 3(in_channels)*1*3*3+1 + 1*1024*1*1+1024 ≈ 2,076パラメータしかなく
# 全体(約35.2M)の0.006%程度なので、このクラスはMediSegEdgeAttUNetと
# ほぼ完全にパラメータ数が一致する("U-Net + MHSA, no positional
# encoding"に相当、EdgeAttNet論文のablationと同じ切り分け)。
# report/main.texが指摘してきた「edge priorそのものの効果か、単に
# attention容量が増えただけか」という交絡を分離するための唯一の差分。


class PlainMHSA(nn.Module):
    """edge priorなしの標準Multi-Head Self-Attention(bottleneck上)。

    EdgeGuidedMHSAと同じchannel数・head数・残差接続+LayerNormを保ちつつ、
    Q・Kへの edge_prior 加算だけを行わない(Q=K=V=bottleneck特徴量)。
    """

    def __init__(self, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        attn_out, _ = self.mhsa(x_flat, x_flat, x_flat)  # Q=K=V=x(edge priorなし)
        attn_out = self.norm(attn_out + x_flat)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegPlainMHSAUNet(nn.Module):
    """MediSegEdgeAttUNetからedge priorだけを除いた matched-parameter 対照群。

    入出力インターフェースはMediSegUNet/MediSegEdgeAttUNetと完全に同一。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.plain_attn = PlainMHSA(bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.plain_attn(x5)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)

"""1-to-1 Multi-detector bipartite matching and continuous score/mask fusion (Idea 8, v132).

Fixes three structural issues in ``run_fold_oof_eval_consensus.py`` (v120/v121c/v124c):
1. Replaces unconstrained transitive union-find with greedy descending-IoU 1-to-1 bipartite
   clustering (at most 1 proposal per detector in every cluster), preventing adjacent-filament
   chaining across detectors.
2. Replaces hard step-function vote counting (``count(s >= 0.26) >= 2``, which discards
   ``(0.45, 0.24, 0)``, ``(0.24, 0.24, 0.24)``, and solo ``(0.75, 0, 0)``) with continuous
   calibrated cluster scoring ``sum_d w_d * s_d`` (missing detector contributes ``0.0``).
3. Supports score-weighted multi-detector proposal mask fusion before ``MediSegEdgeAttUNet``.
"""


from dataclasses import dataclass

import numpy as np
from pycocotools import mask as mask_utils


@dataclass(frozen=True)
class DetectorProposal:
    det_name: str
    conf: float
    raw_rle: dict
    prop_idx: int


@dataclass
class MatchedCluster:
    members: dict[str, DetectorProposal]  # det_name -> DetectorProposal (strictly <=1 per det)

    def score_of(self, det_name: str) -> float:
        p = self.members.get(det_name)
        return float(p.conf) if p is not None else 0.0

    def weighted_score(self, weights: dict[str, float]) -> float:
        return float(sum(w * self.score_of(d) for d, w in weights.items()))

    def max_conf(self) -> float:
        return max((float(p.conf) for p in self.members.values()), default=0.0)

    def count_above(self, thr: float) -> int:
        return sum(1 for p in self.members.values() if float(p.conf) >= thr)

    def best_member(self) -> DetectorProposal:
        return max(self.members.values(), key=lambda p: (float(p.conf), p.det_name))


def match_detector_proposals_1to1(
    proposals: list[DetectorProposal],
    *,
    iou_threshold: float = 0.45,
) -> list[MatchedCluster]:
    """Group cross-detector proposals by descending mask IoU with <=1 member per detector."""
    n = len(proposals)
    if n == 0:
        return []
    rles = [p.raw_rle for p in proposals]
    iou_mat = np.asarray(mask_utils.iou(rles, rles, [0] * n), dtype=np.float64)

    parent = list(range(n))
    det_sets = [{proposals[i].det_name} for i in range(n)]

    def find(x: int) -> int:
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:
            parent[x], x = root, parent[x]
        return root

    pairs: list[tuple[float, int, int]] = []
    for i in range(n):
        di = proposals[i].det_name
        for j in range(i + 1, n):
            if di == proposals[j].det_name:
                continue
            v = float(iou_mat[i, j])
            if v >= iou_threshold:
                pairs.append((v, i, j))
    pairs.sort(key=lambda t: (-t[0], t[1], t[2]))

    for _, i, j in pairs:
        ri, rj = find(i), find(j)
        if ri != rj and det_sets[ri].isdisjoint(det_sets[rj]):
            parent[rj] = ri
            det_sets[ri] |= det_sets[rj]

    grouped: dict[int, dict[str, DetectorProposal]] = {}
    for idx in range(n):
        r = find(idx)
        p = proposals[idx]
        grouped.setdefault(r, {})[p.det_name] = p

    clusters = [MatchedCluster(members=m) for m in grouped.values()]
    clusters.sort(key=lambda c: -c.max_conf())
    return clusters


def fuse_cluster_proposal_mask(
    cluster: MatchedCluster,
    *,
    mode: str = "max_conf",
    vote_threshold: float = 0.45,
) -> np.ndarray:
    """Build bool proposal mask for ``MediSegEdgeAttUNet`` from a matched cluster."""
    if mode == "max_conf" or len(cluster.members) == 1:
        best = cluster.best_member()
        return mask_utils.decode(best.raw_rle) != 0
    if mode != "soft_vote":
        raise ValueError(f"Unknown mask fusion mode: {mode}")

    acc: np.ndarray | None = None
    w_sum = 0.0
    for p in cluster.members.values():
        w = max(float(p.conf), 1e-4)
        m = (mask_utils.decode(p.raw_rle) != 0).astype(np.float32)
        acc = (w * m) if acc is None else (acc + w * m)
        w_sum += w
    assert acc is not None
    return (acc / max(w_sum, 1e-6)) >= float(vote_threshold)


CropSpec = tuple[int, int, int, int, np.ndarray, float, tuple[int, int]]


def rle_to_crop_spec(rle: dict) -> CropSpec:
    """Decode a mask RLE once into a compact bounding-box crop tuple (~6 KB vs 4.2 MB)."""
    m = mask_utils.decode(rle) != 0
    h, w = m.shape
    x, y, bw, bh = [round(float(v)) for v in mask_utils.toBbox(rle)]
    if bw <= 0 or bh <= 0:
        return (0, 0, 0, 0, np.zeros((0, 0), dtype=bool), 0.0, (h, w))
    x0, y0 = max(0, x), max(0, y)
    x1, y1 = min(w, x + bw), min(h, y + bh)
    sub = np.ascontiguousarray(m[y0:y1, x0:x1])
    return (y0, y1, x0, x1, sub, float(sub.sum()), (h, w))


def fuse_refined_crop_specs(
    cluster: MatchedCluster,
    specs_by_det: dict[str, list[CropSpec]],
    *,
    mode: str = "best_member",
) -> CropSpec:
    """Fuse already-refined ``CropSpec`` masks across members of a 1-to-1 matched cluster.

    Modes:
    - ``"best_member"``: return the refined ``CropSpec`` of ``cluster.best_member()``.
    - ``"majority_3plus"``: when ``>=3`` non-empty members agree on an instance, take pixelwise
      majority across their refined crops (unweighted ``>=(K+1)//2`` for odd ``K``, confidence-
      weighted ``>=0.5*sum(w)`` for even ``K`` to break 2-vs-2 ties cleanly); fall back to
      ``best_member`` when ``<=2`` members match so thin filament tips are never clipped by
      2-way strict intersection.
    - ``"soft_w50"``: confidence-weighted pixel average ``>= 0.5 * sum(w)`` for ``>=2`` members.
    """
    bm = cluster.best_member()
    bm_spec = specs_by_det[bm.det_name][bm.prop_idx]
    if mode == "best_member" or len(cluster.members) == 1:
        return bm_spec

    member_specs: list[CropSpec] = []
    weights: list[float] = []
    for dname, prop in cluster.members.items():
        sp = specs_by_det[dname][prop.prop_idx]
        if sp[5] > 0.0:
            member_specs.append(sp)
            weights.append(max(float(prop.conf), 1e-4))
    k = len(member_specs)
    if k == 0:
        return bm_spec
    if k == 1 or (mode == "majority_3plus" and k < 3):
        return bm_spec if bm_spec[5] > 0.0 else member_specs[0]

    hw = member_specs[0][6]
    y0 = min(sp[0] for sp in member_specs)
    y1 = max(sp[1] for sp in member_specs)
    x0 = min(sp[2] for sp in member_specs)
    x1 = max(sp[3] for sp in member_specs)
    acc = np.zeros((y1 - y0, x1 - x0), dtype=np.float32)

    if mode == "majority_3plus" and (k % 2 == 1):
        for sy0, sy1, sx0, sx1, sub, _, _ in member_specs:
            acc[sy0 - y0 : sy1 - y0, sx0 - x0 : sx1 - x0] += sub.astype(np.float32)
        fused = acc >= float((k + 1) // 2)
    elif mode in ("majority_3plus", "soft_w50"):
        w_sum = float(sum(weights))
        for (sy0, sy1, sx0, sx1, sub, _, _), w in zip(member_specs, weights):
            acc[sy0 - y0 : sy1 - y0, sx0 - x0 : sx1 - x0] += w * sub.astype(np.float32)
        fused = acc >= (0.50 * w_sum)
    else:
        raise ValueError(f"Unknown refined crop fusion mode: {mode}")

    ys, xs = np.nonzero(fused)
    if len(xs) == 0:
        return (0, 0, 0, 0, np.zeros((0, 0), dtype=bool), 0.0, hw)
    ny0, ny1 = int(ys.min()), int(ys.max()) + 1
    nx0, nx1 = int(xs.min()), int(xs.max()) + 1
    sub_f = np.ascontiguousarray(fused[ny0:ny1, nx0:nx1])
    return (y0 + ny0, y0 + ny1, x0 + nx0, x0 + nx1, sub_f, float(len(xs)), hw)


def fast_apply_own_specs(
    specs: list[CropSpec],
    thr: float = 0.90,
    min_area: int = 20,
) -> list[dict]:
    """BBox-cropped ownership matching ``ownership.apply_ownership`` (drop_contained_then_area_asc)."""
    n = len(specs)
    if n == 0:
        return []
    h, w = specs[0][6]
    dropped: set[int] = set()
    for i in range(n):
        for j in range(i + 1, n):
            if i in dropped or j in dropped:
                continue
            y0i, y1i, x0i, x1i, sub_i, ai, _ = specs[i]
            y0j, y1j, x0j, x1j, sub_j, aj, _ = specs[j]
            if ai <= 0.0 and aj <= 0.0:
                continue
            smaller = i if ai <= aj else j
            smaller_area = min(ai, aj)
            iy0, iy1 = max(y0i, y0j), min(y1i, y1j)
            ix0, ix1 = max(x0i, x0j), min(x1i, x1j)
            if iy1 <= iy0 or ix1 <= ix0 or (iy1 - iy0) * (ix1 - ix0) < thr * max(smaller_area, 1.0):
                continue
            inter = int(
                np.logical_and(
                    sub_i[iy0 - y0i : iy1 - y0i, ix0 - x0i : ix1 - x0i],
                    sub_j[iy0 - y0j : iy1 - y0j, ix0 - x0j : ix1 - x0j],
                ).sum()
            )
            if inter / max(smaller_area, 1.0) >= thr:
                dropped.add(smaller)

    keep = [i for i in range(n) if i not in dropped]
    if not keep:
        return []
    occ = np.zeros((h, w), dtype=bool)
    kept_subs: list[tuple[int, int, int, int, np.ndarray]] = []
    for idx in sorted(keep, key=lambda i: specs[i][5]):
        y0, y1, x0, x1, sub, _, _ = specs[idx]
        if sub.size == 0:
            continue
        ex = sub & ~occ[y0:y1, x0:x1]
        if int(ex.sum()) >= min_area:
            occ[y0:y1, x0:x1] |= ex
            kept_subs.append((y0, y1, x0, x1, ex))
    if not kept_subs:
        return []
    canvas = np.zeros((h, w, len(kept_subs)), dtype=np.uint8, order="F")
    for k, (y0, y1, x0, x1, ex) in enumerate(kept_subs):
        canvas[y0:y1, x0:x1, k] = ex
    return mask_utils.encode(canvas)

In [ ]:
# [3/4] GT 読込 & prod / yolo26 キャッシュ構築 (既存なら Drive から即時ロード)
def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sparse_label_pq(sum_iou, tp, fp, fn, r=0.77):
    den = r * tp + 0.5 * fp + 0.5 * (1.0 - r) * tp + 0.5 * r * fn
    return float(r * sum_iou / den) if den > 0 else 0.0

TRAIN_JSON = next(DATA_CACHE.rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
coco = json.loads(TRAIN_JSON.read_text(encoding="utf-8"))
images = {im["id"]: im for im in coco["images"]}
annotations = defaultdict(list)
for ann in coco["annotations"]:
    annotations[ann["image_id"]].append(ann)
file_to_ids = defaultdict(list)
for iid, im in images.items():
    file_to_ids[im["file_name"]].append(iid)
train_files = sorted(file_to_ids)

folds_path = ARTIFACT_ROOT / "grouped_folds.csv"
assert file_sha256(folds_path) == GROUPED_FOLDS_SHA256
fold_df = pd.read_csv(folds_path)
fold_of_file = dict(zip(fold_df["file_name"], fold_df["fold"].astype(int)))
eval_files = {f: [n for n in train_files if fold_of_file[n] == f] for f in FOLDS}
all_files = eval_files[0] + eval_files[2]
assert len(eval_files[0]) == 236 and len(eval_files[2]) == 235

def to_rle(ann, h, w):
    seg = ann["segmentation"]
    if isinstance(seg, list):
        return mask_utils.merge(mask_utils.frPyObjects(seg, h, w))
    if isinstance(seg.get("counts"), list):
        return mask_utils.frPyObjects(seg, h, w)
    rle = dict(seg)
    if isinstance(rle["counts"], str):
        rle["counts"] = rle["counts"].encode("utf-8")
    return rle

gt_rles = {
    iid: [to_rle(a, int(im["height"]), int(im["width"])) for a in annotations[iid] if not a.get("iscrowd", 0)]
    for iid, im in images.items()
}

def sq_box(mask, context=1.65, minimum=96):
    h, w = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, w, h
    cx, cy = (xs.min() + xs.max() + 1) / 2.0, (ys.min() + ys.max() + 1) / 2.0
    side = min(max(int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context)), minimum), min(h, w))
    x0, y0 = round(cx - side / 2.0), round(cy - side / 2.0)
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > w: x0 -= x1 - w; x1 = w
    if y1 > h: y0 -= y1 - h; y1 = h
    return max(0, x0), max(0, y0), min(w, x1), min(h, y1)

def refine_batch(refiner, img_bgr, masks):
    if not masks:
        return []
    boxes = [sq_box(m) for m in masks]
    crops = np.stack([
        cv2.resize(img_bgr[y0:y1, x0:x1], (320, 320), interpolation=cv2.INTER_LINEAR).transpose(2, 0, 1)
        for x0, y0, x1, y1 in boxes
    ], 0)
    t = torch.from_numpy(crops).float().cuda() / 255.0
    outs = []
    with torch.no_grad():
        for i in range(0, len(t), 32):
            outs.append(torch.sigmoid(refiner(t[i:i + 32])).squeeze(1).cpu().numpy())
    probs = np.concatenate(outs, 0)
    res = []
    for m, (x0, y0, x1, y1), pr in zip(masks, boxes, probs):
        pb = cv2.resize(pr, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR) >= 0.45
        fm = np.zeros_like(m, bool)
        fm[y0:y1, x0:x1] = pb
        res.append(fm & m)
    return res

def ensure_base_cache(fold, det_key):
    out_pkl = V136_CACHE_DIR / f"{det_key}_f{fold}_best_cache.pkl"
    if out_pkl.exists():
        data = pickle.loads(out_pkl.read_bytes())
        print(f"再利用: {out_pkl.name} ({len(data)} obs)")
        return {row[0]: row for row in data}

    assert torch.cuda.is_available(), "初回キャッシュ作成には GPU が必要です"
    for name in eval_files[fold]:
        dst = LOCAL_PREPROC / name
        if not dst.exists():
            shutil.copy2(PREPROC_CACHE / name, dst)

    refiner = MediSegEdgeAttUNet(3, 1, False).cuda().eval()
    ck = torch.load(BASE_CKPTS[fold]["refiner"], map_location="cuda", weights_only=False)
    refiner.load_state_dict(ck.get("best_model") or ck["model"], strict=True)
    del ck

    model = YOLO(str(BASE_CKPTS[fold][det_key]))
    cache_rows = []
    for name in tqdm(eval_files[fold], desc=f"f{fold}_{det_key} 推論+refine"):
        ip = LOCAL_PREPROC / name
        r = model.predict(str(ip), imgsz=1280, conf=0.08, iou=0.55, agnostic_nms=True, retina_masks=True, verbose=False)[0]
        h, w = r.orig_shape
        ms = []
        if r.masks is not None:
            for d in r.masks.data.detach().cpu().numpy():
                ms.append((cv2.resize(d, (w, h), interpolation=cv2.INTER_LINEAR) if d.shape != (h, w) else d) >= 0.5)
        cf = r.boxes.conf.detach().cpu().numpy().astype(np.float32) if r.boxes is not None else np.zeros(0, np.float32)
        vi = [i for i, x in enumerate(ms) if float(x.sum()) >= 20.0]
        vms = [ms[i] for i in vi]
        vcf = cf[vi] if vi else np.zeros(0, np.float32)
        areas = [float(x.sum()) for x in vms]
        im = cv2.imread(str(ip), cv2.IMREAD_COLOR)
        rms = refine_batch(refiner, im, vms)
        raw_rles = [mask_utils.encode(np.asfortranarray(x.astype(np.uint8))) for x in vms]
        ref_rles = [mask_utils.encode(np.asfortranarray(x.astype(np.uint8))) for x in rms]
        cache_rows.append((name, vcf, areas, raw_rles, ref_rles))
    del model, refiner
    torch.cuda.empty_cache()
    out_pkl.write_bytes(pickle.dumps(cache_rows))
    print(f"保存完了: {out_pkl.name}")
    return {row[0]: row for row in cache_rows}

def load_oof_pkl(pkl_path):
    rows = pickle.loads(pkl_path.read_bytes())
    return {row[0]: row for row in rows}

DET_CACHES = {0: {}, 2: {}}
for f in FOLDS:
    DET_CACHES[f]["prod"] = ensure_base_cache(f, "prod")
    DET_CACHES[f]["yolo26"] = ensure_base_cache(f, "yolo26")
    DET_CACHES[f]["tal_best"] = load_oof_pkl(V133_CACHE_DIR / f"small_tal_f{f}_best_cache.pkl")
    if has_v133_last:
        DET_CACHES[f]["tal_last"] = load_oof_pkl(V133_CACHE_DIR / f"small_tal_f{f}_last_cache.pkl")
    if has_v134_best:
        DET_CACHES[f]["sptal_best"] = load_oof_pkl(V134_CACHE_DIR / f"spine_small_tal_f{f}_best_cache.pkl")
    if has_v134_last:
        DET_CACHES[f]["sptal_last"] = load_oof_pkl(V134_CACHE_DIR / f"spine_small_tal_f{f}_last_cache.pkl")
    if has_v138_best:
        DET_CACHES[f]["sp_best"] = load_oof_pkl(V138_CACHE_DIR / f"spine_aux_f{f}_best_cache.pkl")
    if has_v138_last:
        DET_CACHES[f]["sp_last"] = load_oof_pkl(V138_CACHE_DIR / f"spine_aux_f{f}_last_cache.pkl")

print("ロード済み検出器キー:", list(DET_CACHES[0].keys()))


In [ ]:
# [4/4] 27倍高速 BBox-Cropped RLE 排他化 (consensus_fusion 同梱) & 2-Fold 交差盲検採点
# 全検出器・全画像の refined RLE を一度だけ軽量 BBox crop spec 化 (~6KB/個)
REF_SPECS = {0: {}, 2: {}}
for f in FOLDS:
    for dkey, cmap in DET_CACHES[f].items():
        REF_SPECS[f][dkey] = {
            name: [rle_to_crop_spec(r) for r in row[4]]
            for name, row in cmap.items()
        }

def score_obs_spec_predictions(specs_by_file, files_subset):
    s_iou = 0.0; tp = 0; fp = 0; fn = 0; obs = {}
    for name in files_subset:
        pr = fast_apply_own_specs(specs_by_file[name])
        os_s = [0.0, 0, 0, 0]
        for iid in file_to_ids[name]:
            tr = gt_rles[iid]
            if not tr:
                fp += len(pr); os_s[2] += len(pr)
            elif not pr:
                fn += len(tr); os_s[3] += len(tr)
            else:
                mat = np.asarray(mask_utils.iou(pr, tr, [0] * len(tr)), np.float64).T
                hits = mat > 0.5
                si = float(mat[hits].sum()); ti = int(hits.sum())
                fpi = int((hits.sum(0) == 0).sum()); fni = int((hits.sum(1) == 0).sum())
                s_iou += si; tp += ti; fp += fpi; fn += fni
                os_s[0] += si; os_s[1] += ti; os_s[2] += fpi; os_s[3] += fni
        obs[name] = tuple(os_s)
    den = tp + 0.5 * fp + 0.5 * fn
    return {
        "pq": float(s_iou / den) if den > 0 else 0.0,
        "pq_sparse_r077": sparse_label_pq(s_iou, tp, fp, fn, 0.77),
        "sq": float(s_iou / tp) if tp else 0.0,
        "rq": float(tp / den) if den > 0 else 0.0,
        "tp": tp, "fp": fp, "fn": fn,
    }, obs

def pool_metrics_from_obs(obs_dict, files_subset):
    arr = np.array([obs_dict[f] for f in files_subset], np.float64)
    s_iou, tp, fp, fn = arr.sum(0)
    tp, fp, fn = round(tp), round(fp), round(fn)
    den = tp + 0.5 * fp + 0.5 * fn
    return {
        "pq": float(s_iou / den) if den > 0 else 0.0,
        "pq_sparse_r077": sparse_label_pq(s_iou, tp, fp, fn, 0.77),
        "sq": float(s_iou / tp) if tp else 0.0,
        "rq": float(tp / den) if den > 0 else 0.0,
        "tp": tp, "fp": fp, "fn": fn,
    }

def paired_boot(obs_base, obs_cand, files_subset, sparse=False):
    rng = np.random.default_rng(3407)
    ba = np.array([obs_base[f] for f in files_subset], np.float64)
    ca = np.array([obs_cand[f] for f in files_subset], np.float64)
    idx = rng.integers(0, len(files_subset), size=(2000, len(files_subset)))
    bs = ba[idx].sum(1); cs = ca[idx].sum(1)
    if sparse:
        r = 0.77
        bden = r * bs[:, 1] + 0.5 * bs[:, 2] + 0.5 * (1 - r) * bs[:, 1] + 0.5 * r * bs[:, 3]
        cden = r * cs[:, 1] + 0.5 * cs[:, 2] + 0.5 * (1 - r) * cs[:, 1] + 0.5 * r * cs[:, 3]
        d = (r * cs[:, 0]) / cden - (r * bs[:, 0]) / bden
    else:
        d = cs[:, 0] / (cs[:, 1] + 0.5 * cs[:, 2] + 0.5 * cs[:, 3]) - bs[:, 0] / (bs[:, 1] + 0.5 * bs[:, 2] + 0.5 * bs[:, 3])
    return {
        "n_obs": len(files_subset),
        "mean_delta": float(d.mean()),
        "ci95_low": float(np.percentile(d, 2.5)),
        "ci95_high": float(np.percentile(d, 97.5)),
        "prob_positive": float((d > 0).mean()),
    }

# 1対1 クラスタリングは active detector 集合ごとに1回だけ計算してメモリ保持
CLUSTER_CACHE = {}
def get_preclustered(fold, det_tuple):
    key = (fold, det_tuple)
    if key in CLUSTER_CACHE:
        return CLUSTER_CACHE[key]
    by_file = {}
    for name in eval_files[fold]:
        props = []
        for d in det_tuple:
            _, cf, areas, raw_rles, _ = DET_CACHES[fold][d][name]
            for i in range(len(cf)):
                if areas[i] >= 20.0:
                    props.append(DetectorProposal(d, float(cf[i]), raw_rles[i], i))
        by_file[name] = match_detector_proposals_1to1(props, iou_threshold=0.45)
    CLUSTER_CACHE[key] = by_file
    return by_file

# 評価する Arm 群を構築
ARMS = [
    ("prod", {"prod": 1.0}, 0.26, None),
    ("yolo26_026", {"yolo26": 1.0}, 0.26, None),
    ("tal_best_026", {"tal_best": 1.0}, 0.26, None),
    ("tal_best_028", {"tal_best": 1.0}, 0.28, None),
    ("pair_prod_tal_b_t21", {"prod": 0.50, "tal_best": 0.50}, 0.21, None),
    ("pair_prod_tal_b_t23", {"prod": 0.50, "tal_best": 0.50}, 0.23, None),
    ("trio_b_equal_t20", {"prod": 1/3, "yolo26": 1/3, "tal_best": 1/3}, 0.20, None),
    ("trio_b_equal_t22", {"prod": 1/3, "yolo26": 1/3, "tal_best": 1/3}, 0.22, None),
    ("trio_b_tal45_y30_z25_t21", {"tal_best": 0.45, "prod": 0.30, "yolo26": 0.25}, 0.21, None),
    ("trio_b_tal45_y30_z25_t23", {"tal_best": 0.45, "prod": 0.30, "yolo26": 0.25}, 0.23, None),
    ("cons2of3_trio_b_c26", {"prod": 1.0, "yolo26": 1.0, "tal_best": 1.0}, 0.26, 2),
    ("cons2of3_trio_b_c26_maj3", {"prod": 1.0, "yolo26": 1.0, "tal_best": 1.0}, 0.26, 2, "majority_3plus"),
]

if has_v133_last:
    ARMS.extend([
        ("tal_last_026", {"tal_last": 1.0}, 0.26, None),
        ("tal_last_028", {"tal_last": 1.0}, 0.28, None),
        ("pair_prod_tal_l_t21", {"prod": 0.50, "tal_last": 0.50}, 0.21, None),
        ("pair_prod_tal_l_t23", {"prod": 0.50, "tal_last": 0.50}, 0.23, None),
        ("trio_l_equal_t21", {"prod": 1/3, "yolo26": 1/3, "tal_last": 1/3}, 0.21, None),
        ("trio_l_tal45_y30_z25_t21", {"tal_last": 0.45, "prod": 0.30, "yolo26": 0.25}, 0.21, None),
        ("trio_l_tal45_y30_z25_t23", {"tal_last": 0.45, "prod": 0.30, "yolo26": 0.25}, 0.23, None),
        ("cons2of3_trio_l_c26", {"prod": 1.0, "yolo26": 1.0, "tal_last": 1.0}, 0.26, 2),
        ("cons2of3_trio_l_c26_maj3", {"prod": 1.0, "yolo26": 1.0, "tal_last": 1.0}, 0.26, 2, "majority_3plus"),
    ])

if has_v134_best:
    ARMS.extend([
        ("sptal_best_026", {"sptal_best": 1.0}, 0.26, None),
        ("sptal_best_028", {"sptal_best": 1.0}, 0.28, None),
        ("pair_tal_sptal_b_t22", {"tal_best": 0.50, "sptal_best": 0.50}, 0.22, None),
        ("pair_tal_sptal_b_t24", {"tal_best": 0.50, "sptal_best": 0.50}, 0.24, None),
        ("cons2of3_sptal_prod_z_c26_bm", {"sptal_best": 1.0, "prod": 1.0, "yolo26": 1.0}, 0.26, 2, "best_member"),
        ("cons2of3_sptal_prod_z_c26_maj3", {"sptal_best": 1.0, "prod": 1.0, "yolo26": 1.0}, 0.26, 2, "majority_3plus"),
        ("quad_b_t20", {"sptal_best": 0.35, "tal_best": 0.30, "prod": 0.20, "yolo26": 0.15}, 0.20, None),
        ("quad_b_t22", {"sptal_best": 0.35, "tal_best": 0.30, "prod": 0.20, "yolo26": 0.15}, 0.22, None),
        ("quad_b_t24", {"sptal_best": 0.35, "tal_best": 0.30, "prod": 0.20, "yolo26": 0.15}, 0.24, None),
        ("cons2of4_quad_b_c26", {"sptal_best": 1.0, "tal_best": 1.0, "prod": 1.0, "yolo26": 1.0}, 0.26, 2),
        ("cons2of4_quad_b_c26_maj3", {"sptal_best": 1.0, "tal_best": 1.0, "prod": 1.0, "yolo26": 1.0}, 0.26, 2, "majority_3plus"),
    ])

if has_v134_last:
    ARMS.extend([
        ("sptal_last_026", {"sptal_last": 1.0}, 0.26, None),
        ("pair_tal_sptal_l_t22", {"tal_last": 0.50, "sptal_last": 0.50}, 0.22, None),
        ("quad_l_t21", {"sptal_last": 0.35, "tal_last": 0.30, "prod": 0.20, "yolo26": 0.15}, 0.21, None),
        ("quad_l_t23", {"sptal_last": 0.35, "tal_last": 0.30, "prod": 0.20, "yolo26": 0.15}, 0.23, None),
    ])

if has_v138_best:
    ARMS.extend([
        ("sp_best_026", {"sp_best": 1.0}, 0.26, None),
        ("sp_best_028", {"sp_best": 1.0}, 0.28, None),
        ("pair_spb_prod_t23", {"sp_best": 0.60, "prod": 0.40}, 0.23, None),
        ("trio_spb50_y25_z25_t22", {"sp_best": 0.50, "prod": 0.25, "yolo26": 0.25}, 0.22, None),
        ("trio_spb50_y25_z25_t24", {"sp_best": 0.50, "prod": 0.25, "yolo26": 0.25}, 0.24, None),
        ("cons2of3_spb_prod_z_c26_maj3", {"sp_best": 1.0, "prod": 1.0, "yolo26": 1.0}, 0.26, 2, "majority_3plus"),
        ("cons2of4_spb_talb_prod_z_c26_maj3", {"sp_best": 1.0, "tal_best": 1.0, "prod": 1.0, "yolo26": 1.0}, 0.26, 2, "majority_3plus"),
    ])
    if has_v133_last:
        ARMS.extend([
            ("pair_spb55_tall45_t22", {"sp_best": 0.55, "tal_last": 0.45}, 0.22, None),
            ("pair_spb55_tall45_t24", {"sp_best": 0.55, "tal_last": 0.45}, 0.24, None),
            ("quad_spb40_tall25_y20_z15_t22", {"sp_best": 0.40, "tal_last": 0.25, "prod": 0.20, "yolo26": 0.15}, 0.22, None),
            ("quad_spb40_tall25_y20_z15_t24", {"sp_best": 0.40, "tal_last": 0.25, "prod": 0.20, "yolo26": 0.15}, 0.24, None),
            ("cons2of4_spb_tall_prod_z_c26_maj3", {"sp_best": 1.0, "tal_last": 1.0, "prod": 1.0, "yolo26": 1.0}, 0.26, 2, "majority_3plus"),
        ])
    if has_v134_best:
        ARMS.extend([
            ("pair_spb55_sptalb45_t24", {"sp_best": 0.55, "sptal_best": 0.45}, 0.24, None),
        ])

results = {}
obs_by_arm = {}

for arm_spec in tqdm(ARMS, desc="全 Arm 2-Fold 採点"):
    arm_name, wdict, thr, dvote = arm_spec[:4]
    mmode = arm_spec[4] if len(arm_spec) >= 5 else "best_member"
    det_tuple = tuple(sorted(wdict.keys()))
    obs_merged = {}
    fold_metrics = {}
    for f in FOLDS:
        if len(det_tuple) == 1:
            d0 = det_tuple[0]
            pm_specs = {}
            for name in eval_files[f]:
                _, cf, areas, _, _ = DET_CACHES[f][d0][name]
                sp_list = REF_SPECS[f][d0][name]
                pm_specs[name] = [
                    sp_list[i]
                    for i in range(len(cf))
                    if cf[i] >= thr and areas[i] >= 20.0
                ]
        else:
            cl_map = get_preclustered(f, det_tuple)
            pm_specs = {}
            for name in eval_files[f]:
                s_by_det = {d: REF_SPECS[f][d][name] for d in det_tuple}
                chosen = []
                for cl in cl_map[name]:
                    ok = (cl.count_above(thr) >= dvote) if dvote is not None else (cl.weighted_score(wdict) >= thr)
                    if ok:
                        sp = fuse_refined_crop_specs(cl, s_by_det, mode=mmode)
                        if sp[5] >= 20.0:
                            chosen.append(sp)
                pm_specs[name] = chosen
        fm, fo = score_obs_spec_predictions(pm_specs, eval_files[f])
        fold_metrics[f] = fm
        obs_merged.update(fo)
    obs_by_arm[arm_name] = obs_merged
    results[arm_name] = {
        "weights": wdict,
        "threshold": thr,
        "discrete_vote": dvote,
        "mask_mode": mmode,
        "fold0": fold_metrics[0],
        "fold2": fold_metrics[2],
        "pooled_471": pool_metrics_from_obs(obs_merged, all_files),
    }

prod_obs = obs_by_arm["prod"]
p0 = results["prod"]["fold0"]["pq"]
p2 = results["prod"]["fold2"]["pq"]
pp = results["prod"]["pooled_471"]["pq"]
ps = results["prod"]["pooled_471"]["pq_sparse_r077"]

print(f"\n=== v136 2-FOLD HONEST SUMMARY (471 obs, v134_included={has_v134_best}) ===")
for arm_name, r in results.items():
    bt_pq = paired_boot(prod_obs, obs_by_arm[arm_name], all_files, sparse=False)
    bt_sp = paired_boot(prod_obs, obs_by_arm[arm_name], all_files, sparse=True)
    r["bootstrap_pq_vs_prod"] = bt_pq
    r["bootstrap_sparse_vs_prod"] = bt_sp
    f0d = r["fold0"]["pq"] - p0
    f2d = r["fold2"]["pq"] - p2
    ppd = r["pooled_471"]["pq"] - pp
    spd = r["pooled_471"]["pq_sparse_r077"] - ps
    print(
        f"{arm_name:25s} | f0={r['fold0']['pq']:.5f} ({f0d:+.5f}) | f2={r['fold2']['pq']:.5f} ({f2d:+.5f}) "
        f"| pool={r['pooled_471']['pq']:.5f} ({ppd:+.5f}) | sparse={r['pooled_471']['pq_sparse_r077']:.5f} ({spd:+.5f}) "
        f"| FP={r['pooled_471']['fp']:4d} | P(>0)={bt_pq['prob_positive']:.3f}"
    )

cand_names = [k for k in results if k != "prod"]
sel_f2_pq = max(cand_names, key=lambda k: results[k]["fold2"]["pq"])
sel_f0_pq = max(cand_names, key=lambda k: results[k]["fold0"]["pq"])
sel_f2_sp = max(cand_names, key=lambda k: results[k]["fold2"]["pq_sparse_r077"])
sel_f0_sp = max(cand_names, key=lambda k: results[k]["fold0"]["pq_sparse_r077"])

cross_selection = {
    "by_pq": {
        "selected_on_fold2": {
            "arm": sel_f2_pq,
            "fold2_delta": results[sel_f2_pq]["fold2"]["pq"] - p2,
            "blind_fold0_delta": results[sel_f2_pq]["fold0"]["pq"] - p0,
            "pooled_delta": results[sel_f2_pq]["pooled_471"]["pq"] - pp,
            "pooled_sparse_delta": results[sel_f2_pq]["pooled_471"]["pq_sparse_r077"] - ps,
            "bootstrap_pq": results[sel_f2_pq]["bootstrap_pq_vs_prod"],
        },
        "selected_on_fold0": {
            "arm": sel_f0_pq,
            "fold0_delta": results[sel_f0_pq]["fold0"]["pq"] - p0,
            "blind_fold2_delta": results[sel_f0_pq]["fold2"]["pq"] - p2,
            "pooled_delta": results[sel_f0_pq]["pooled_471"]["pq"] - pp,
            "pooled_sparse_delta": results[sel_f0_pq]["pooled_471"]["pq_sparse_r077"] - ps,
            "bootstrap_pq": results[sel_f0_pq]["bootstrap_pq_vs_prod"],
        },
    },
    "by_sparse_r077": {
        "selected_on_fold2": {
            "arm": sel_f2_sp,
            "fold2_sparse_delta": results[sel_f2_sp]["fold2"]["pq_sparse_r077"] - results["prod"]["fold2"]["pq_sparse_r077"],
            "blind_fold0_sparse_delta": results[sel_f2_sp]["fold0"]["pq_sparse_r077"] - results["prod"]["fold0"]["pq_sparse_r077"],
            "pooled_sparse_delta": results[sel_f2_sp]["pooled_471"]["pq_sparse_r077"] - ps,
            "pooled_pq_delta": results[sel_f2_sp]["pooled_471"]["pq"] - pp,
            "bootstrap_sparse": results[sel_f2_sp]["bootstrap_sparse_vs_prod"],
        },
        "selected_on_fold0": {
            "arm": sel_f0_sp,
            "fold0_sparse_delta": results[sel_f0_sp]["fold0"]["pq_sparse_r077"] - results["prod"]["fold0"]["pq_sparse_r077"],
            "blind_fold2_sparse_delta": results[sel_f0_sp]["fold2"]["pq_sparse_r077"] - results["prod"]["fold2"]["pq_sparse_r077"],
            "pooled_sparse_delta": results[sel_f0_sp]["pooled_471"]["pq_sparse_r077"] - ps,
            "pooled_pq_delta": results[sel_f0_sp]["pooled_471"]["pq"] - pp,
            "bootstrap_sparse": results[sel_f0_sp]["bootstrap_sparse_vs_prod"],
        },
    },
}

print("\n=== CROSS-FOLD BLIND SELECTION ===")
print(json.dumps(cross_selection, indent=2, ensure_ascii=False))

manifest = {
    "experiment_id": "v136_strong_fusion_2fold",
    "has_v133_best": has_v133_best,
    "has_v133_last": has_v133_last,
    "has_v134_best": has_v134_best,
    "has_v134_last": has_v134_last,
    "has_v138_best": has_v138_best,
    "has_v138_last": has_v138_last,
    "results": results,
    "cross_selection": cross_selection,
}
mp = ARTIFACT_ROOT / "v136_strong_fusion_2fold_manifest.json"
with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=ARTIFACT_ROOT, suffix=".tmp", delete=False) as h:
    h.write(json.dumps(manifest, indent=2, ensure_ascii=False))
    h.flush()
    os.fsync(h.fileno())
    tmp_p = h.name
os.replace(tmp_p, mp)
print("Saved manifest:", mp)
